# paragami: folding and flattening parameters

paragami describes the *shape and constraints* of a parameter with a **pattern**. A pattern moves a value between three representations:

| representation | what it is | example for a covariance matrix |
|---|---|---|
| **folded** | the value as you'd write it in code (an array, or a dict of arrays) | a $K \times K$ positive definite matrix |
| **flat** | the folded value's entries as one vector, constraints and all | its $K^2$ entries |
| **free** | an unconstrained vector of the right dimension | the $K(K+1)/2$ entries of the log-Cholesky factor |

Conversions are named `<from>_to_<to>`: `folded_to_free`, `free_to_folded`, `folded_to_flat` and `flat_to_folded`. The generic `flatten(value, to_free=...)` and `fold(vector, from_free=...)` choose the vector representation with a keyword whose prefix says which side it describes.

The free representation is what optimizers and samplers want. By default paragami uses Stan's unconstraining transforms, so free vectors and log Jacobians match Stan's unconstrained space exactly. Where TensorFlow Probability uses a different transform, `bijector="tfp"` selects TFP's instead. Everything is pure JAX, so it works under `grad`, `jit` and `vmap`.

Run `./setup_venv.sh` in this folder once and register the kernel it prints; this notebook uses that kernel ("Python (modelgami)").

In [1]:
import arviz as az
import jax
import jax.numpy as jnp
import numpy as np
import paragami as pg

jax.config.update("jax_enable_x64", True)

## A single pattern

A positive scalar, and a vector bounded to $(0, 1)$.

In [2]:
sigma_pattern = pg.NumericScalarPattern(lb=0.0)
x = sigma_pattern.folded_to_free(2.0)
print("free value:", x, "  (log 2 =", np.log(2.0), ")")
print("folded back:", sigma_pattern.free_to_folded(x))
print("flat value:", sigma_pattern.folded_to_flat(2.0))

p_pattern = pg.NumericVectorPattern(3, lb=0.0, ub=1.0)
p = jnp.array([0.1, 0.5, 0.9])
print("free value of p:", p_pattern.folded_to_free(p), " (the logit)")

# The same conversions with the generic methods:
print(p_pattern.flatten(p, to_free=True), p_pattern.fold(p_pattern.flatten(p, to_free=True), from_free=True))

free value: [0.69314718]   (log 2 = 0.6931471805599453 )
folded back: 2.0
flat value: [2.]
free value of p: [-2.19722458  0.          2.19722458]  (the logit)


[-2.19722458  0.          2.19722458] [0.1 0.5 0.9]


## Structured constraints

Some patterns have a free dimension smaller than their flat dimension. A simplex in $K$ dimensions has $K - 1$ free parameters, and a $K \times K$ covariance matrix has $K(K+1)/2$. There is a pattern for each of Stan's constrained types.

In [3]:
patterns = {
    "simplex(4)": pg.SimplexArrayPattern(4),
    "ordered(3)": pg.OrderedVectorPattern(3),
    "positive_ordered(3)": pg.OrderedVectorPattern(3, positive=True),
    "cov_matrix(3)": pg.PSDSymmetricMatrixPattern(3),
    "cholesky_factor_cov(3)": pg.CholeskyCovPattern(3),
    "cholesky_factor_corr(3)": pg.CholeskyCorrPattern(3),
    "corr_matrix(3)": pg.CorrMatrixPattern(3),
    "unit_vector(3)": pg.UnitVectorPattern(3),
    "sum_to_zero_vector(4)": pg.SumToZeroVectorPattern(4),
    "sum_to_zero_matrix(3, 4)": pg.SumToZeroMatrixPattern(3, 4),
    "column_stochastic(3, 2)": pg.StochasticMatrixPattern(3, 2, kind="column"),
    "vector<offset=1, multiplier=2>": pg.NumericVectorPattern(3, offset=1.0, multiplier=2.0),
}
for name, pat in patterns.items():
    print(f"{name:25s} flat length {pat.get_flat_length():2d}   free length {pat.get_free_length():2d}")

simplex(4)                flat length  4   free length  3
ordered(3)                flat length  3   free length  3
positive_ordered(3)       flat length  3   free length  3
cov_matrix(3)             flat length  9   free length  6
cholesky_factor_cov(3)    flat length  9   free length  6
cholesky_factor_corr(3)   flat length  9   free length  3
corr_matrix(3)            flat length  9   free length  3
unit_vector(3)            flat length  3   free length  3
sum_to_zero_vector(4)     flat length  4   free length  3
sum_to_zero_matrix(3, 4)  flat length 12   free length  6
column_stochastic(3, 2)   flat length  6   free length  4
vector<offset=1, multiplier=2> flat length  3   free length  3


In [4]:
cov_pattern = pg.PSDSymmetricMatrixPattern(3)
u = jax.random.normal(jax.random.PRNGKey(0), (cov_pattern.get_free_length(),))
cov = cov_pattern.free_to_folded(u)    # any free vector folds to a valid covariance matrix
print(cov)
print("eigenvalues:", jnp.linalg.eigvalsh(cov))
print("round trip error:", jnp.max(jnp.abs(cov_pattern.folded_to_free(cov) - u)))

[[ 0.66253342 -0.63876917  0.15289781]
 [-0.63876917 38.41072605  0.34974222]
 [ 0.15289781  0.34974222  0.51692897]]


eigenvalues: [ 0.40961455  0.75586342 38.42471047]
round trip error: 2.7755575615628914e-17


Patterns where Stan and TFP use different transforms take `bijector="stan"` (the default) or `bijector="tfp"`: one-sided bounds (TFP uses softplus instead of exp), simplexes, and the covariance and correlation patterns. The folded value is the same kind of object either way; only the free coordinates differ.

In [5]:
for bijector in ("stan", "tfp"):
    pat = pg.NumericScalarPattern(lb=0.0, bijector=bijector)
    print(f"{bijector}: free value of 2.0 is {float(pat.folded_to_free(2.0)[0]):.4f}")

stan: free value of 2.0 is 0.6931
tfp: free value of 2.0 is 1.8546


## Dictionaries of parameters

`PatternDict` combines patterns, and its folded values are dicts. This is how a model's parameters are described in modelgami.

In [6]:
pattern = pg.PatternDict(
    mu=pg.NumericVectorPattern(3),
    sigma=pg.NumericScalarPattern(lb=0.0),
    w=pg.SimplexArrayPattern(4),
    cov=pg.PSDSymmetricMatrixPattern(2),
)
params = pattern.draw_random(jax.random.PRNGKey(1))
params

{'mu': Array([-1.18428442, -0.11617041,  0.17269028], dtype=float64),
 'sigma': Array(2.6046731, dtype=float64),
 'w': Array([0.15956632, 0.51823804, 0.12339456, 0.19880108], dtype=float64),
 'cov': Array([[ 0.21674056, -0.02357817],
        [-0.02357817,  0.06943873]], dtype=float64)}

In [7]:
x_free = pattern.folded_to_free(params)
x_flat = pattern.folded_to_flat(params)
print("free length:", x_free.shape[0], "  flat length:", x_flat.shape[0])
print(pattern.get_free_names())

free length: 10   flat length: 12
['mu[0]', 'mu[1]', 'mu[2]', 'sigma', 'w[0]', 'w[1]', 'w[2]', 'cov[0]', 'cov[1]', 'cov[2]']


### Nested dictionaries

A `PatternDict` can contain other `PatternDict`s, to any depth. The folded value is then a nested dict, and the flat and free vectors concatenate everything in order. Entry names show the path.

In [8]:
nested = pg.PatternDict(
    likelihood=pg.PatternDict(beta=pg.NumericVectorPattern(2), sigma=pg.NumericScalarPattern(lb=0.0)),
    mixture=pg.PatternDict(
        weights=pg.SimplexArrayPattern(3),
        components=pg.PatternDict(means=pg.OrderedVectorPattern(3), cov=pg.PSDSymmetricMatrixPattern(2)),
    ),
)
nested_params = nested.draw_random(jax.random.PRNGKey(5))
print(jax.tree_util.tree_map(jnp.shape, nested_params))

u = nested.folded_to_free(nested_params)
print("free length:", nested.get_free_length(), "  flat length:", nested.get_flat_length())
print(nested.get_free_names())
print("round trip error:", jax.tree_util.tree_reduce(
    max, jax.tree_util.tree_map(lambda a, b: float(jnp.max(jnp.abs(a - b))), nested.free_to_folded(u), nested_params)))

{'likelihood': {'beta': (2,), 'sigma': ()}, 'mixture': {'components': {'cov': (2, 2), 'means': (3,)}, 'weights': (3,)}}


free length: 11   flat length: 13
['likelihood.beta[0]', 'likelihood.beta[1]', 'likelihood.sigma', 'mixture.weights[0]', 'mixture.weights[1]', 'mixture.components.means[0]', 'mixture.components.means[1]', 'mixture.components.means[2]', 'mixture.components.cov[0]', 'mixture.components.cov[1]', 'mixture.components.cov[2]']
round trip error: 1.1102230246251565e-16


### Writing a pattern for a structure you already have

If you already have your parameters as a nested structure of dicts, lists and tuples of arrays, you don't need to write the pattern by hand. `make_unconstrained_pattern_like` builds an unconstrained pattern of the same shape. Every pattern can print the code that builds it with `generate_code()`, so together they give code ready to paste into your own code and edit.

Dict keys become the pattern's names. List and tuple entries have no names, so they are named from `default_name` (or the enclosing dict key) with an index, and they fold to dicts.

In [9]:
my_params = {
    "beta": jnp.zeros(3),
    "sigma": 1.0,
    "groups": [jnp.zeros(2), jnp.zeros(2)],
    "nested": {"cov": jnp.eye(2)},
}
print(pg.make_unconstrained_pattern_like(my_params).generate_code())

pg.PatternDict(
    beta=pg.NumericVectorPattern(3),
    sigma=pg.NumericScalarPattern(),
    groups=pg.PatternDict(
        groups_0=pg.NumericVectorPattern(2),
        groups_1=pg.NumericVectorPattern(2),
    ),
    nested=pg.PatternDict(
        cov=pg.NumericArrayPattern((2, 2)),
    ),
)


Paste the printed code and edit it to add constraints. Here `sigma` becomes positive and `cov` a covariance matrix:

In [10]:
my_pattern = pg.PatternDict(
    beta=pg.NumericVectorPattern(3),
    sigma=pg.NumericScalarPattern(lb=0.0),
    groups=pg.PatternDict(
        groups_0=pg.NumericVectorPattern(2),
        groups_1=pg.NumericVectorPattern(2),
    ),
    nested=pg.PatternDict(
        cov=pg.PSDSymmetricMatrixPattern(2),
    ),
)
inferred = pg.make_unconstrained_pattern_like(my_params)
print("free length:", my_pattern.get_free_length(), "  (unconstrained version:", inferred.get_free_length(), ")")
print("generated code rebuilds the inferred pattern exactly:", eval(inferred.generate_code()) == inferred)

free length: 11   (unconstrained version: 12 )
generated code rebuilds the inferred pattern exactly: True


In [11]:
# For a tuple or list at the top level, names come from default_name.
print(pg.make_unconstrained_pattern_like((jnp.zeros(2), 0.0), default_name="theta").get_free_names())

['theta_0[0]', 'theta_0[1]', 'theta_1']


## Log Jacobians

A density over the constrained parameters becomes a density over the free vector by adding the log absolute determinant of the free-to-constrained Jacobian. `free_to_folded` does not compute it; `free_to_folded_log_abs_det` computes only the log determinant, and `free_to_folded_with_log_det` returns both. Here we check it against autodiff for the simplex, where the free and flat dimensions differ, so we use the flat value without its last (redundant) entry.

In [12]:
simplex = pg.SimplexArrayPattern(4)
u = jnp.array([0.3, -1.0, 0.5])
w, log_det = simplex.free_to_folded_with_log_det(u)

J = jax.jacfwd(lambda v: simplex.fold(v, from_free=True)[:-1])(u)
print("paragami log det:", log_det)
print("autodiff log det:", jnp.linalg.slogdet(J)[1])

paragami log det: -5.60889129649251
autodiff log det: -5.60889129649251


## Validation

`validate_folded` checks keys, shapes and constraint values and returns `(ok, message)`. The conversions validate their inputs; inside `jit`, value checks are skipped automatically and shape checks remain.

In [13]:
bad = dict(params, sigma=-1.0)
print(pattern.validate_folded(bad))
print(pattern.validate_folded(dict(params, w=jnp.ones(3) / 3)))
try:
    pattern.flatten(bad, to_free=True)
except Exception as e:
    print(type(e).__name__, ":", e)

(False, "PatternDict(patterns={'mu': ('NumericVectorPattern', {'shape': (3,), 'lb': -inf, 'ub': inf, 'offset': 0.0, 'multiplier': 1.0, 'order': 'C', 'bijector': 'stan'}), 'sigma': ('NumericScalarPattern', {'shape': (), 'lb': 0.0, 'ub': inf, 'offset': 0.0, 'multiplier': 1.0, 'order': 'C', 'bijector': 'stan'}), 'w': ('SimplexArrayPattern', {'simplex_size': 4, 'array_shape': (), 'bijector': 'stan'}), 'cov': ('PSDSymmetricMatrixPattern', {'size': 2, 'bijector': 'stan'})}): ['sigma'] values must be > lb = 0.0 (min -1.0)")
(False, "PatternDict(patterns={'mu': ('NumericVectorPattern', {'shape': (3,), 'lb': -inf, 'ub': inf, 'offset': 0.0, 'multiplier': 1.0, 'order': 'C', 'bijector': 'stan'}), 'sigma': ('NumericScalarPattern', {'shape': (), 'lb': 0.0, 'ub': inf, 'offset': 0.0, 'multiplier': 1.0, 'order': 'C', 'bijector': 'stan'}), 'w': ('SimplexArrayPattern', {'simplex_size': 4, 'array_shape': (), 'bijector': 'stan'}), 'cov': ('PSDSymmetricMatrixPattern', {'size': 2, 'bijector': 'stan'})}): ['w

Conversions in an inner loop can skip every check. `disable_validation()` turns them off for a pattern and, for a `PatternDict` or `PatternArray`, for everything inside it; `enable_validation()` turns them back on.

In [14]:
pattern.disable_validation()
print("no check, no error:", pattern.flatten(bad, to_free=True)[:4])
pattern.enable_validation()
print("children re-enabled too:", pattern["sigma"].validate)

no check, no error: [-1.18428442 -0.11617041  0.17269028         nan]
children re-enabled too: True


## Arrays of patterns

`PatternArray` repeats a pattern over an array of shape `array_shape`, for example one simplex per group.

In [15]:
group_weights = pg.PatternArray((5,), pg.SimplexArrayPattern(3))
print("free length:", group_weights.get_free_length())
W = group_weights.draw_random(jax.random.PRNGKey(2))
print(W.shape, W.sum(axis=-1))

free length: 10


(5, 3) [1. 1. 1. 1. 1.]


## Using patterns with JAX transformations

A function of folded parameters becomes a function of the free vector with `function_input_folded_to_free`. You can then differentiate, compile and vectorize it.

In [16]:
def objective(params):
    return jnp.sum(params["mu"] ** 2) + params["sigma"] + jnp.sum(params["w"] * jnp.arange(4.0)) + jnp.trace(params["cov"])

f = pg.function_input_folded_to_free(objective, patterns=pattern)
print("f(x):", f(x_free))
print("grad:", jax.jit(jax.grad(f))(x_free))

xs = jax.random.normal(jax.random.PRNGKey(3), (10, x_free.shape[0]))
print("vmap over 10 free vectors:", jax.vmap(f)(xs).shape)

f(x): 5.698129887361914


grad: [-2.36856884 -0.23234082  0.34538056  2.6046731  -0.02116473 -0.22949165
 -0.37614301  0.43348113 -0.10129078  0.13374755]


vmap over 10 free vectors: (10,)


A pattern acts on a single value, so batches of draws are folded with `jax.vmap`. The result is a dict of arrays with a leading batch axis. Notebook 3 uses this to turn MCMC output into structured draws.

In [17]:
folded_batch = jax.vmap(lambda v: pattern.fold(v, from_free=True))(xs)
{k: v.shape for k, v in folded_batch.items()}

{'cov': (10, 2, 2), 'mu': (10, 3), 'sigma': (10,), 'w': (10, 4)}

## Draws and ArviZ

MCMC output usually comes as an array of free vectors with shape (chain, draw, free length). paragami converts such arrays to [ArviZ](https://python.arviz.org) and back (install ArviZ, or `paragami[arviz]`):

- `free_draws_to_arviz` and `folded_draws_to_arviz` make ArviZ data with one variable per pattern entry, in constrained form. Nested dict entries are named with dots, such as `"inner.w"`.
- `arviz_to_free_draws` and `arviz_to_folded_draws` read them back.
- `free_draws_to_folded`, `flat_draws_to_folded`, `folded_draws_to_free` and `folded_draws_to_flat` convert between representations without ArviZ.

Here the "draws" are just random free vectors standing in for sampler output.

In [18]:
free_draws = jax.random.normal(jax.random.PRNGKey(6), (4, 500, pattern.get_free_length()))
idata = pg.free_draws_to_arviz(pattern=pattern, draws=free_draws, coords={"component": ["a", "b", "c", "d"]},
                               dims={"w": ["component"]})
az.summary(idata, var_names=["sigma", "w"])

,mean,sd,eti89_lb,eti89_ub,ess_bulk,ess_tail,r_hat,mcse_mean,mcse_sd
sigma,1.6,2,0.2,5,2118,2058,1.00,0.044,0.14
w[a],0.25,0.19,0.035,0.63,1957,1812,1.00,0.0044,0.0034
w[b],0.247,0.188,0.033,0.62,2075,2099,1.00,0.0041,0.0031
w[c],0.255,0.187,0.035,0.62,1845,1922,1.00,0.0044,0.0033
w[d],0.248,0.19,0.034,0.62,2076,1973,1.00,0.0042,0.0032


In [19]:
back = pg.arviz_to_free_draws(pattern=pattern, data=idata)
print("round trip error:", float(jnp.max(jnp.abs(back - free_draws))))

folded = pg.arviz_to_folded_draws(pattern=pattern, data=idata)
print({k: v.shape for k, v in folded.items()})

round trip error: 2.0694557179012918e-13
{'mu': (4, 500, 3), 'sigma': (4, 500), 'w': (4, 500, 4), 'cov': (4, 500, 2, 2)}
